# Colab VLA Server Spike v0

Runs a temporary FastAPI VLA inference server in this Colab runtime and exposes it over an HTTPS tunnel (ngrok or cloudflared), so a **local** `RealVLAPolicyClient` (in the `physical-ai-recycling-cell` repo) can call it.

**This notebook never touches a robot.** It only serves `/health` and `/predict`. The local machine still owns the camera, PyBullet, `SafetySupervisor`, `RobotBackend`, and episode recording -- see `docs/colab_vla_server_spike.md` for the full role split.

**This is a spike/test environment, not a production server.** The session ends when Colab disconnects (idle timeout, 12/24h cap, or you closing the tab), and the tunnel URL is invalidated with it -- you'll need to re-run this notebook and update the local config again next time.

**Default flow in this notebook is a real OpenVLA `openvla-dryrun` load from a Google Drive cache** (sections 1-3b below prepare that). If you only want to check that the tunnel/HTTP plumbing works at all, skip straight to `SERVER_MODE = "mock-action"` in section 4 -- it needs none of the Drive/OpenVLA setup. Either way, run the cells in order top to bottom.

## 1. Runtime / GPU check

Reports what's actually available in this Colab runtime -- torch install, CUDA, GPU name/VRAM, Python version -- and prints a `recommended_mode` so you don't guess. A CPU-only or no-GPU runtime is completely fine for `health-only`/`mock-action` (the two modes this spike's success criteria actually depend on); only `openvla-dryrun` needs a GPU, and even then it degrades gracefully if one isn't available.

In [ ]:
import platform
import subprocess

print(f"python_version: {platform.python_version()}")

try:
    import torch
    torch_installed = True
    torch_version = torch.__version__
    cuda_available = torch.cuda.is_available()
except ImportError:
    torch_installed = False
    torch_version = None
    cuda_available = False

print(f"torch_installed: {torch_installed}")
print(f"torch_version: {torch_version}")
print(f"cuda_available: {cuda_available}")

gpu_name = None
vram_total_gb = None
vram_free_gb = None
if cuda_available:
    gpu_name = torch.cuda.get_device_name(0)
    free_bytes, total_bytes = torch.cuda.mem_get_info(0)
    vram_total_gb = round(total_bytes / (1024 ** 3), 2)
    vram_free_gb = round(free_bytes / (1024 ** 3), 2)

print(f"gpu_name: {gpu_name}")
print(f"vram_total_gb: {vram_total_gb}")
print(f"vram_free_gb: {vram_free_gb}")

if cuda_available and (vram_free_gb or 0) >= 16:
    recommended_mode = "openvla-dryrun (GPU + enough free VRAM detected -- still expect this to be tight for a 7B VLA)"
elif cuda_available:
    recommended_mode = "mock-action (GPU detected but free VRAM looks too low for a 7B model -- openvla-dryrun will likely report model_status=load_failed, which is fine)"
else:
    recommended_mode = "mock-action or health-only (no CUDA GPU in this runtime -- openvla-dryrun will report model_status=load_failed by design, not crash)"

print(f"recommended_mode: {recommended_mode}")

## 2. Install dependencies

Base packages needed regardless of server mode -- torch/transformers are left alone here (see section 2b for those) since reinstalling either can churn for several minutes and isn't needed for `health-only`/`mock-action`.

In [ ]:
%pip install -q fastapi uvicorn pyngrok pillow requests nest_asyncio

### 2b. Install OpenVLA dependencies

Needed for the Drive-cache / `openvla-dryrun` flow this notebook defaults to. Skip this cell (and the rest of section 3b) entirely if you only want `health-only`/`mock-action` -- jump straight to section 4 and set `SERVER_MODE = "mock-action"` instead.

This pins `transformers`/`tokenizers`/`timm` to the exact versions OpenVLA's own model code expects (its `trust_remote_code=True` custom modeling code is version-sensitive), and adds `hf_transfer` for faster Hugging Face Hub downloads.

In [ ]:
%pip uninstall -y transformers tokenizers timm
%pip install -q -r https://raw.githubusercontent.com/openvla/openvla/main/requirements-min.txt
%pip install -q huggingface_hub hf_transfer

### 2c. Validate OpenVLA imports

Confirms the pinned versions from 2b actually import cleanly before you spend time on the Drive download in section 3b.

In [ ]:
import transformers
print("transformers:", transformers.__version__)

from transformers import AutoProcessor, AutoModelForVision2Seq
print("OpenVLA imports OK")

## 3. Clone the repo

The actual server (`openvla_server_real/colab_vla_server.py`) lives in the `physical-ai-recycling-cell` repo, not duplicated inline here -- this cell just clones/pulls the repo and puts it on `sys.path`, so the notebook and the local codebase never drift apart.

**Do not import `colab_vla_server` yet.** That import happens in section 4, only after the Drive cache (section 3b) is ready and `SERVER_MODE`/`OPENVLA_MODEL_PATH`/`OPENVLA_LOCAL_FILES_ONLY` are set -- importing it earlier wouldn't download anything by itself (the server is lazy-load, see section 4), but setting up env vars first keeps the notebook's cell order matching what actually needs to happen in what order.

In [ ]:
import os
import sys

REPO_URL = "REPLACE_WITH_YOUR_REPO_URL"  # TODO: replace with your GitHub repo URL, e.g. https://github.com/<you>/physical-ai-recycling-cell.git
REPO_DIR = "/content/physical-ai-recycling-cell"

if not os.path.isdir(REPO_DIR):
    !git clone "$REPO_URL" "$REPO_DIR"
else:
    !cd "$REPO_DIR" && git pull

sys.path.insert(0, REPO_DIR)

## 3b. Prepare OpenVLA Drive cache

OpenVLA-7B is ~15GB -- downloading it fresh every Colab session is slow and wasteful, and a session that disconnects mid-download means starting over from 0%. This section downloads it **once** into Google Drive (permanent storage that survives across sessions), then copies it into this session's local Colab disk (`/content/openvla-7b`, faster reads for actual inference than Drive's network filesystem).

Skip this whole section if you only want `mock-action`/`health-only` -- it's only needed for a real `openvla-dryrun` model load.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

Hugging Face login -- raises your download rate limit and is required for some gated/mirrored repos.

In [ ]:
from huggingface_hub import login
login()

Download the full snapshot to Drive. `resume_download=True` means a dropped Colab session can pick back up instead of restarting from 0% -- just re-run this cell.

In [ ]:
import os
from huggingface_hub import snapshot_download

os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

DRIVE_MODEL_DIR = "/content/drive/MyDrive/openvla-cache/openvla-7b"

repo_path = snapshot_download(
    repo_id="openvla/openvla-7b",
    local_dir=DRIVE_MODEL_DIR,
    local_dir_use_symlinks=False,
    resume_download=True,
    max_workers=1,
)

print("downloaded to:", repo_path)

Sanity-check what actually landed in Drive (expect several multi-GB `.safetensors` shards).

In [ ]:
!du -sh /content/drive/MyDrive/openvla-cache/openvla-7b
!ls -lh /content/drive/MyDrive/openvla-cache/openvla-7b | head -80
!ls -lh /content/drive/MyDrive/openvla-cache/openvla-7b | grep safetensors

Copy from Drive (slower network filesystem) to this session's local Colab disk (faster reads for actual model loading/inference).

In [ ]:
!rm -rf /content/openvla-7b
!mkdir -p /content/openvla-7b
!rsync -ah --info=progress2 /content/drive/MyDrive/openvla-cache/openvla-7b/ /content/openvla-7b/

Confirm the local copy is actually usable by `transformers` with **no network call at all** (`local_files_only=True`) before wiring it into the server.

In [ ]:
from transformers import AutoProcessor

MODEL_PATH = "/content/openvla-7b"

processor = AutoProcessor.from_pretrained(
    MODEL_PATH,
    trust_remote_code=True,
    local_files_only=True,
)

print("processor loaded from local cache")

## 4. Choose a server mode and import the app

```
health-only     no model loaded, ever; /predict always fails with model_not_loaded (503).
                Use this first to validate the tunnel/HTTP plumbing alone.
mock-action     no real model; reuses DummyOpenVLAPolicy to return a deterministic,
                safe 7-DoF action. Needs none of section 3b's Drive setup --
                if you only want to check connectivity, set SERVER_MODE =
                "mock-action" below and skip straight here from section 1.
openvla-dryrun  (this notebook's default) uses the local Drive cache prepared in
                section 3b via OPENVLA_MODEL_PATH + OPENVLA_LOCAL_FILES_ONLY=1,
                so /load_model (section 6b) loads from local disk instead of
                re-downloading from Hugging Face. Importing the server below
                NEVER downloads or loads OpenVLA regardless of mode -- that
                only ever happens later, explicitly, via POST /load_model.
```

`openvla-direct` (applying raw OpenVLA output straight to the robot) is intentionally not implemented anywhere in this repo.

In [ ]:
import os

SERVER_MODE = "openvla-dryrun"  # "health-only" | "mock-action" | "openvla-dryrun"
os.environ["COLAB_VLA_SERVER_MODE"] = SERVER_MODE

# Use the local copy made from Google Drive cache (section 3b).
# This prevents /load_model from trying to download OpenVLA from Hugging Face again.
os.environ["OPENVLA_MODEL_PATH"] = "/content/openvla-7b"
os.environ["OPENVLA_LOCAL_FILES_ONLY"] = "1"

# This import is always fast (well under a second) and NEVER downloads or
# loads OpenVLA, regardless of SERVER_MODE -- model loading only ever
# happens later, lazily, via POST /load_model (section 6b). If you set
# SERVER_MODE = "mock-action" instead, the two OPENVLA_* env vars above are
# simply unused -- no Drive cache or /load_model call is needed for that mode.
from openvla_server_real.colab_vla_server import app, SERVER_MODE as loaded_mode
print(f"server_mode: {loaded_mode}")

## 5. Start the server + public tunnel

Runs uvicorn in a background thread (so this cell returns and the notebook stays usable), then opens a public HTTPS tunnel to it. Uses ngrok if `NGROK_AUTHTOKEN` is set (Colab: `Secrets` panel or `os.environ`); otherwise falls back to cloudflared's no-account quick tunnel.

In [ ]:
import threading
import time

import nest_asyncio
import uvicorn

nest_asyncio.apply()

LOCAL_PORT = 9000

config = uvicorn.Config(app, host="127.0.0.1", port=LOCAL_PORT, log_level="info")
server = uvicorn.Server(config)

server_thread = threading.Thread(target=server.run, daemon=True)
server_thread.start()
time.sleep(2)
print(f"local server running on http://127.0.0.1:{LOCAL_PORT}")

In [ ]:
import os

ngrok_authtoken = os.environ.get("NGROK_AUTHTOKEN")
public_url = None

if ngrok_authtoken:
    from pyngrok import ngrok

    ngrok.set_auth_token(ngrok_authtoken)
    tunnel = ngrok.connect(LOCAL_PORT, "http")
    public_url = tunnel.public_url
    print(f"tunnel: ngrok ({public_url})")
else:
    print("NGROK_AUTHTOKEN not set -- falling back to cloudflared (no account needed).")
    print("Install once per session: !wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared && chmod +x cloudflared")
    print(f"Then run in the background: !./cloudflared tunnel --url http://127.0.0.1:{LOCAL_PORT}")
    print("cloudflared prints its own https://<random>.trycloudflare.com URL to stdout/logs -- copy that as public_url below.")

if public_url:
    health_url = f"{public_url}/health"
    predict_url = f"{public_url}/predict"
    print(f"health_url: {health_url}")
    print(f"predict_url: {predict_url}")

## 6. Test curl commands

Run these from a local terminal (not this notebook) to sanity-check the tunnel before pointing `RealVLAPolicyClient` at it.

Expected `/health` response right after section 4/5 (before `/load_model` in 6b), with `SERVER_MODE = "openvla-dryrun"` and the Drive cache env vars from section 4:

```json
{
  "status": "ok",
  "server_mode": "openvla-dryrun",
  "model_status": "not_loaded",
  "model_status_reason": "model load has not been requested",
  "model_id_or_path": "/content/openvla-7b",
  "local_files_only": true,
  "version": "v0"
}
```

`model_id_or_path` being `/content/openvla-7b` (not `openvla/openvla-7b`) confirms the Drive cache env vars from section 4 actually took effect -- check this *before* running section 6b, since that's the whole point of preparing the cache.

In [ ]:
if public_url:
    print("# From your local machine:\n")
    print(f"curl {public_url}/health\n")
    print(
        f"curl -X POST {public_url}/predict \\\n"
        "  -H \"Content-Type: application/json\" \\\n"
        "  -d '{\"instruction\": \"test\", \"target_object_position\": [0.4, -0.1, 0.05], \"bin_position\": [0.3, 0.35, 0.05]}'"
    )
else:
    print("Set public_url first (see cell 5's cloudflared fallback instructions).")

## 6b. Trigger OpenVLA loading (only for `openvla-dryrun`)

Skip this entirely for `health-only`/`mock-action`. The server has been up and answering `/health` since section 5 -- this cell is the *only* thing that ever attempts to actually load OpenVLA into memory, and it's fully separate from starting the server.

**This is where the real OpenVLA load happens.** If the Drive cache (section 3b) is intact, this should load from `/content/openvla-7b` and **not** re-trigger a Hugging Face shard download -- watch the logs; if you see download progress bars here, the local cache is missing files (re-run the `snapshot_download`/`rsync` cells in 3b).

How to read the result:

- **`model_status: "loaded"`** -- success. `/predict` will now return real (raw, non-executable -- see section 4's mode table) OpenVLA output.
- **`model_status: "load_failed"`, reason mentions `CUDA out of memory`** -- the download/cache worked fine; this is a Colab GPU VRAM limit, not a data problem. Try a smaller/quantized checkpoint or a bigger GPU tier.
- **`model_status: "load_failed"`, reason mentions a missing file/shard** -- the Drive cache is incomplete. Re-run the `snapshot_download` cell in 3b (it resumes, so this is cheap) and then the `rsync` copy cell again.
- **Either way, the server keeps running** -- `/health` and `/predict` both stay reachable, and `/predict` keeps responding a structured `model_not_loaded`/`action_adapter_required` error (so your local `RealVLAPolicyClient` just falls back) instead of the whole server going down. Calling this cell more than once is safe -- an in-progress or already-finished load is reported back as-is instead of being restarted.

In [ ]:
import requests

if SERVER_MODE == "openvla-dryrun":
    response = requests.post(f"http://127.0.0.1:{LOCAL_PORT}/load_model", timeout=600)
    print(response.json())
    print()
    print("Same thing from your local machine once you have the public URL:")
    print("curl -X POST <predict_url without /predict>/load_model")
else:
    print(f"SERVER_MODE={SERVER_MODE!r} -- /load_model is only meaningful for openvla-dryrun, skipping.")

## 7. Update the local config and probe from your local machine

```bash
python scripts/update_colab_vla_config.py \
  --base-url <public_url printed above> \
  --config configs/real_vla_backend_colab_config.json

python -m benchmark.probe_colab_vla_server \
  --real-vla-config configs/real_vla_backend_colab_config.json \
  --with-image
```

See `docs/colab_vla_server_spike.md` for the full local probe / full-demo walkthrough, and its "OpenVLA Drive cache workflow" section for more on the Drive/local-cache split above.

## 8. When you're done: Colab session/shutdown notes

- **The tunnel URL dies with the Colab session.** Idle timeout, the 12/24h session cap, or just closing the tab all invalidate it immediately -- your local `RealVLAPolicyClient` will start failing its health check (and fall back to `local-dummy`/`fastapi-dummy`, if configured, which is the expected/safe behavior).
- **Re-running this notebook issues a brand-new URL.** ngrok free tier and cloudflared quick tunnels do not preserve the previous URL across restarts.
- **You must re-run `scripts/update_colab_vla_config.py` with the new URL** every time you restart this notebook -- there is no automatic sync between Colab and your local config file.
- **The Google Drive cache (section 3b) persists across sessions -- `/content/openvla-7b` does not.** `/content` is this session's local VM disk and is wiped when the session ends; Drive is not. Next session, you only need to re-run the `rsync` copy cell (fast, from Drive) instead of `snapshot_download` again (slow, from Hugging Face) -- unless you want to check for a model update.
- **`/load_model` (section 6b) is independent of the server/tunnel lifecycle.** The server can be up (and `/health` reachable) for the entire session without ever calling it -- only call it when you actually want to attempt a real OpenVLA load.
- This was never meant to run unattended for a long session -- see `docs/colab_vla_server_spike.md` for why (and what a real deployment would use instead: a persistent server, not a notebook + tunnel).